# Sprint 3 — Process Mining
Análise de processos hospitalares com PM4Py sobre o `gold_event_log`.

**Notebook:** `03_process_mining.ipynb`  
**Pipeline de origem:** `gold_transformations`  
**Schema de leitura:** `hospital_santa_rosa.gold_fluxo`  
**Autor:** Ediney Magalhães  
**Criado em:** 2026-06-15

In [0]:
%pip install pm4py
dbutils.library.restartPython()

In [0]:
import pm4py
import pandas as pd

In [0]:
# leitura do event log canônico fo Unity Catalog como Spark DataFrame
df_spark = spark.table("hospital_santa_rosa.gold_fluxo.gold_event_log")

# confirma volume e schema antes de trazer para memória
print(f'Total de registros: {df_spark.count():,}')
df_spark.printSchema()

In [0]:
# converte Spark DataFrame para Pandas
df_pandas = df_spark.toPandas()

# confirma que a conversão manteve o volume correto
print(f'Registros no Pandas: {len(df_pandas):,}')
print(f'Tipo timestamp: {df_pandas['timestamp'].dtype}')

In [0]:
# declarando timestamp em UTC (PM4PY necessita de timestamp com timezone explícito)
df_pandas['timestamp'] = df_pandas['timestamp'].dt.tz_localize('UTC')

# confirmação do tipo correto
print(f'Tipo timestamp após correção: {df_pandas['timestamp'].dtype}')

In [0]:
# preparando DataFrame para PM4PY, declarando quais colunas corresponde ao conceito XES
df_formatado = pm4py.format_dataframe(
    df_pandas,
    case_id='case_id',
    activity_key='activity',
    timestamp_key='timestamp'
)

# confirmando aplicação 
print(f'Colunas após format_dataframe: {list(df_formatado.columns)}')

In [0]:
# converte o DataFrame pandas anotado para o objeto EventLog hierárquico PM4Py
event_log = pm4py.convert_to_event_log(df_formatado)

# confirma o número de traces (casos) e eventos no EventLog
print(f"Total de traces: {len(event_log):,}")
print(f"Total de eventos: {sum(len(trace) for trace in event_log):,}")

## Descoberta de Processo

In [0]:
# aplicando o algoritmo Inductive Miner
process_tree = pm4py.discover_process_tree_inductive(event_log)

## Variant Analysis

In [0]:
variantes = pm4py.get_variants(event_log)
print(f"Total de variantes distintas: {len(variantes)}")

In [0]:
# converte o dicionário de variantes para uma lista ranqueada por frequência
ranking_variantes = sorted(
    variantes.items(),
    key=lambda item: len(item[1]),
    reverse=True
)

# exibe as 10 variantes mais frequentes
total_casos = len(event_log)
print(f"{'Rank':<6} {'Casos':>6} {'% do Total':>10}  Sequência")
print("-" * 80)
for rank, (sequencia, traces) in enumerate(ranking_variantes[:10], start=1):
    frequencia = len(traces)
    percentual = frequencia / total_casos * 100
    atividades = " → ".join(sequencia)
    print(f"{rank:<6} {frequencia:>6} {percentual:>9.1f}%  {atividades}")

In [0]:
import pandas as pd

# constrói o DataFrame do ranking de variantes
linhas = []
for rank, (sequencia, traces) in enumerate(ranking_variantes, start=1):
    linhas.append({
        "rank": rank,
        "sequencia": " → ".join(sequencia),
        "total_eventos": len(sequencia),
        "total_casos": len(traces),
        "cobertura_perc": round(len(traces) / total_casos * 100, 2),
        "data_referencia": pd.Timestamp.now(tz="UTC").date()
    })

df_variantes = pd.DataFrame(linhas)

# persiste o ranking de variantes no schema gold_fluxo
df_spark_variantes = spark.createDataFrame(df_variantes)

df_spark_variantes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("hospital_santa_rosa.gold_fluxo.gold_variant_analysis")

print(f"Total de variantes: {len(df_variantes):,}")
print("gold_variant_analysis salva com sucesso")